Evaluate whether converting PDFs to Markdown preserves enough document
structure (headings, lists, tables, formulas and hierarchy) to simplify
semantic chunking for the RAG pipeline.

## Environment Setup

Install and import candidate Markdown extraction libraries.

In [ ]:
# import pymupdf
# import pymupdf4llm
# from tqdm import tqdm

## Load Test Document

Use the same mathematics PDF throughout the experiment.

In [ ]:
# pdf = []

# reader = pymupdf.open("./src/pdfs/logical_reasoning.pdf")

# with tqdm(total=len(reader), desc="Converting PDF", unit="page") as pbar:
#     for page_num in range(len(reader)):
#         md = pymupdf4llm.to_markdown(
#             reader,
#             pages=[page_num],      # Convert only one page
#             show_progress=False,   # Disable built-in progress
#             use_ocr=False,
#         )

#         pdf.append(md)
#         pbar.update(1)

# markdown = "\n".join(pdf)

In [ ]:
# print(markdown)

In [ ]:
# from collections import Counter
# import fitz
# import pandas as pd


# def extract_font_statistics(pdf_path):
#     doc = fitz.open(pdf_path)

#     counter = Counter()

#     for page in doc:
#         page_dict = page.get_text("dict")
        

#         for block in page_dict["blocks"]:
            
#             if "lines" not in block:
#                 continue

#             for line in block["lines"]:
#                 for span in line["spans"]:
#                     print(span)
#                     size = round(span["size"], 2)
#                     font = span["font"]
#                     flags = span["flags"]

#                     bold = bool(flags & 16)
#                     italic = bool(flags & 2)

#                     key = (
#                         size,
#                         font,
#                         bold,
#                         italic
#                     )

#                     counter[key] += 1

#     rows = []

#     for (size, font, bold, italic), count in counter.items():
#         rows.append({
#             "font_size": size,
#             "font_name": font,
#             "bold": bold,
#             "italic": italic,
#             "count": count
#         })

#     return pd.DataFrame(rows)

# df = extract_font_statistics('./src/pdfs/logical_reasoning.pdf')

In [ ]:
# size_df = (
#     df.groupby("font_size")
#       .agg(
#           count=("count", "sum"),
#           bold_count=("count", lambda x: x[df.loc[x.index, "bold"]].sum()),
#           italic_count=("count", lambda x: x[df.loc[x.index, "italic"]].sum()),
#           font_variants=("font_name", "nunique")
#       )
#       .reset_index()
# )

# size_df["bold_ratio"] = size_df["bold_count"] / size_df["count"]
# size_df["italic_ratio"] = size_df["italic_count"] / size_df["count"]

# size_df = size_df.sort_values("font_size", ascending=False)

# size_df

In [ ]:
# import matplotlib.pyplot as plt

# plot_df = (
#     df[(df['font_size'] >= 13) & (df['font_size'] <= 42)]
#     .groupby("font_size", as_index=False)["count"]
#     .sum()
#     .sort_values("font_size")
# )

# plt.figure(figsize=(12, 6))

# plt.bar(
#     plot_df["font_size"],
#     plot_df["count"],
#     width=0.5
# )

# plt.xlabel("Font Size")
# plt.ylabel("Count")
# plt.title("Font Size Distribution (≤ 40 pt)")
# plt.xticks(plot_df["font_size"], rotation=45)

# plt.tight_layout()
# plt.show()

In [ ]:
# size_df = size_df.sort_values("font_size", ascending=False).reset_index(drop=True)

# size_df["gap_below"] = (
#     size_df["font_size"] -
#     size_df["font_size"].shift(-1)
# )

# size_df["ratio_below"] = (
#     size_df["font_size"] /
#     size_df["font_size"].shift(-1)
# )

In [ ]:
# size_df[
#     [
#         "font_size",
#         "count",
#         "bold_ratio",
#         "gap_below",
#         "ratio_below"
#     ]
# ]

In [ ]:
# import matplotlib.pyplot as plt

# plot_df = size_df[size_df["font_size"] <= 40].copy()

# plt.figure(figsize=(12, 6))

# plt.bar(
#     plot_df["font_size"],
#     plot_df["gap_below"],
#     width=0.5
# )

# for _, row in plot_df.iterrows():
#     plt.text(
#         row["font_size"],
#         row["gap_below"] + 0.1,
#         f'{row["gap_below"]:.2f}',
#         ha="center"
#     )

# plt.xlabel("Font Size")
# plt.ylabel("Gap to Next Smaller Font Size")
# plt.title("Font Size Gaps")
# plt.xticks(plot_df["font_size"], rotation=45)

# plt.tight_layout()
# plt.show()

In [ ]:

# df[
#     [
#         "page_number",
#         "text",
#         "font_size",
#         "bold_ratio",
#         "word_count",
#         "spacing_before",
#         "spacing_after",
#         "numbering_depth",
#         "x0",
#         "alignment",
#     ]
# ].head(50)



In [ ]:
# # Show relatively large fonts
# candidates = df[
#     (df["font_size"] >= 15) |
#     (df["bold_ratio"] >= 0.8) |
#     (df["has_numbering"])
# ].copy()

# print(
#     candidates[
#         [
#             "page_number",
#             "text",
#             "font_size",
#             "bold_ratio",
#             "word_count",
#             "spacing_before",
#             "spacing_after",
#             "x0",
#             "alignment",
#             "numbering_pattern",
#             "numbering_depth",
#         ]
#     ].to_string(index=False)
# )

In [ ]:
from collections import Counter
from pathlib import Path

import pymupdf  # PyMuPDF
import matplotlib.pyplot as plt
import pandas as pd

BOLD_FLAG = 1 << 4    # bit 4
ITALIC_FLAG = 1 << 1  # bit 1

FILE_PATH ="../src/pdfs/logical_reasoning.pdf"


In [ ]:
def extract_font_spans(pdf_path: str | Path) -> pd.DataFrame:
    """
    Walk every text span in the PDF and count occurrences grouped by
    (font_size, font_name, bold, italic).

    Returns
    -------
    DataFrame with columns: font_size, font_name, bold, italic, count
    """
    doc = pymupdf.open(pdf_path)
    counter: Counter[tuple[float, str, bool, bool]] = Counter()

    for page in doc:
        for block in page.get_text("dict")["blocks"]:
            if "lines" not in block:  # skip image blocks
                continue
            for line in block["lines"]:
                for span in line["spans"]:
                    key = (
                        round(span["size"], 2),
                        span["font"],
                        bool(span["flags"] & BOLD_FLAG),
                        bool(span["flags"] & ITALIC_FLAG),
                    )
                    counter[key] += 1

    doc.close()

    return pd.DataFrame(
        [
            {
                "font_size": size,
                "font_name": font,
                "bold": bold,
                "italic": italic,
                "count": count,
            }
            for (size, font, bold, italic), count in counter.items()
        ]
    )


In [ ]:
span_df = extract_font_spans(FILE_PATH)

In [ ]:
def summarize_by_size(span_df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse span-level stats down to one row per font size:
    total occurrence count, bold/italic usage ratio, font-name diversity,
    and the size gap/ratio down to the next-smaller size (useful for
    detecting natural heading-level cluster boundaries).
    """
    weighted = span_df.assign(
        bold_weighted=span_df["count"] * span_df["bold"],
        italic_weighted=span_df["count"] * span_df["italic"],
    )

    size_df = (
        weighted.groupby("font_size")
        .agg(
            count=("count", "sum"),
            bold_count=("bold_weighted", "sum"),
            italic_count=("italic_weighted", "sum"),
            font_variants=("font_name", "nunique"),
        )
        .reset_index()
        .sort_values("font_size", ascending=False)
        .reset_index(drop=True)
    )

    size_df["bold_ratio"] = size_df["bold_count"] / size_df["count"]
    size_df["italic_ratio"] = size_df["italic_count"] / size_df["count"]
    size_df["gap_below"] = size_df["font_size"] - size_df["font_size"].shift(-1)
    size_df["ratio_below"] = size_df["font_size"] / size_df["font_size"].shift(-1)

    return size_df



In [ ]:
size_df = summarize_by_size(span_df)

In [ ]:
def plot_size_distribution(
    span_df: pd.DataFrame,
    min_size: float = 13.0,
    max_size: float = 42.0,
) -> None:
    """Bar chart of total span count per font size, within [min_size, max_size]."""
    plot_df = (
        span_df[span_df["font_size"].between(min_size, max_size)]
        .groupby("font_size", as_index=False)["count"]
        .sum()
        .sort_values("font_size")
    )

    plt.figure(figsize=(12, 6))
    plt.bar(plot_df["font_size"], plot_df["count"], width=0.5)
    plt.xlabel("Font Size")
    plt.ylabel("Span Count")
    plt.title(f"Font Size Distribution ({min_size}–{max_size} pt)")
    plt.xticks(plot_df["font_size"], rotation=45)
    plt.tight_layout()
    plt.show()


def plot_size_gaps(size_df: pd.DataFrame, max_size: float = 40.0) -> None:
    """Bar chart of the gap to the next-smaller font size, annotated with values."""
    plot_df = size_df[size_df["font_size"] <= max_size].copy()

    plt.figure(figsize=(12, 6))
    plt.bar(plot_df["font_size"], plot_df["gap_below"], width=0.5)

    for _, row in plot_df.iterrows():
        if pd.notna(row["gap_below"]):
            plt.text(
                row["font_size"],
                row["gap_below"] + 0.1,
                f'{row["gap_below"]:.2f}',
                ha="center",
            )

    plt.xlabel("Font Size")
    plt.ylabel("Gap to Next Smaller Font Size")
    plt.title("Font Size Gaps (cluster boundary candidates)")
    plt.xticks(plot_df["font_size"], rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_size_distribution(span_df)
plot_size_gaps(size_df) 

In [ ]:
size_df

In [ ]:
import pandas as pd

def build_dynamic_hdr_info(
    size_df,
    min_occurrence_ratio=0.001,   # candidate must be ≥0.1% of all spans to not be an outlier
    min_gap_ratio=1.05,           # ≥5% size jump = new heading level
    max_heading_levels=6,
    min_bold_ratio=None,          # optional: e.g. 0.5 to require majority-bold styling
):
    """
    Derives heading-level clusters from font-size statistics and returns
    a hdr_info(span, page=None) callable for pymupdf4llm.to_markdown().
    """
    df = size_df.copy().sort_values("font_size", ascending=False).reset_index(drop=True)
    total_count = df["count"].sum()

    # Step 1: body text = size with the highest total occurrence.
    body_size = df.loc[df["count"].idxmax(), "font_size"]

    # Step 2: candidates must be visually larger than body text.
    candidates = df[df["font_size"] > body_size].copy()

    # Step 3: drop outliers -- rare sizes aren't deliberate heading styles.
    min_count = max(2, total_count * min_occurrence_ratio)
    candidates = candidates[candidates["count"] >= min_count]

    # Optional: require the size to be predominantly bold to count as a heading.
    if min_bold_ratio is not None:
        candidates = candidates[candidates["bold_ratio"] >= min_bold_ratio]

    if candidates.empty:
        return lambda span, page=None: ""

    # Step 4: recompute ratio_below on the FILTERED set (neighbors changed
    # after outlier removal), then cluster on dynamic gap thresholds.
    candidates = candidates.sort_values("font_size", ascending=False).reset_index(drop=True)
    candidates["ratio_below"] = candidates["font_size"] / candidates["font_size"].shift(-1)

    cluster_id = 0
    cluster_ids = [0]
    for i in range(1, len(candidates)):
        prev_ratio = candidates.loc[i - 1, "ratio_below"]
        if pd.isna(prev_ratio) or prev_ratio >= min_gap_ratio:
            cluster_id += 1
        cluster_ids.append(cluster_id)
    candidates["cluster"] = cluster_ids

    # Step 5: one markdown level per cluster, largest size = level 1.
    cluster_order = list(dict.fromkeys(candidates["cluster"]))[:max_heading_levels]
    cluster_to_level = {c: i + 1 for i, c in enumerate(cluster_order)}

    size_to_level = {
        round(row["font_size"], 1): cluster_to_level[row["cluster"]]
        for _, row in candidates.iterrows()
        if row["cluster"] in cluster_to_level
    }
    sorted_sizes = sorted(size_to_level.keys(), reverse=True)

    def hdr_info(span, page=None):
        size = round(span["size"], 1)
        for s in sorted_sizes:
            if size >= s - 0.3:   # tolerance for float rendering noise
                return "#" * size_to_level[s] + " "
        return ""

    return hdr_info

In [ ]:
hdr_info = build_dynamic_hdr_info(size_df)

In [ ]:
import pymupdf
import pymupdf4llm

pymupdf4llm.use_layout(False)

doc = pymupdf.open(FILE_PATH)


markdown = pymupdf4llm.to_markdown(
    doc,
    show_progress=True,
    # use_ocr=False,
    hdr_info=hdr_info
)

print(markdown)

## Markdown Inspection

Manually inspect the generated Markdown to evaluate structural preservation.

In [ ]:
markdown

## Evalution

| Property      | Result | Notes |
| ------------- | ------ | ----- |
| Headings      | ✅      |       |
| Lists         | ✅      |       |
| Tables        | ⚠️     |       |
| Formulas      | ✅      |       |
| Reading Order | ✅      |       |
| Images        | ❌      |       |


## Algorithm

```Use structure as a separation
    Each pdf has different heading styles and size. and PDF reader pymupdf4llm reads it and convert it into the markdown. 
    due to variation of writing style it heading hierarchy marker may vary from pdf to pdf.

    but the writing style of the author remains same throughout the PDF. so the PDF gets same hierarchy marker to same level of heading and list.

    so rather than hardcoding. 
    we identify the structure and we group them into the semantic structure.


Tasks
    1) read PDF (if page wise -> more batter)

    2) Identify the structure
        Focus on heading hierarchy. because topics mostly contains the semantic boundaries as a headings of next topic.


In [ ]:
import re

def build_tree(text: str) -> dict:
    """
    Builds a semantic tree from Markdown.

    `depth` represents the node's depth in the semantic tree.

    `heading_level` represents the original Markdown heading level.

    Example:

        # A
        ### B
        ## C
        ### D

    becomes:

        A (depth=0, heading_level=1)
        ├── B (depth=1, heading_level=3)
        └── C (depth=1, heading_level=2)
            └── D (depth=2, heading_level=3)
    """

    root = {
        "text": "",
        "depth": 0,
        "children": [],
    }

    heading_pattern = re.compile(
        r"^(#{1,6})[ \t]+(.+?)\s*$",
        re.MULTILINE,
    )

    matches = list(heading_pattern.finditer(text))

    # No headings
    if not matches:
        root["text"] = text.strip()
        return root

    # Text before the first heading
    preamble = text[:matches[0].start()].strip()

    if preamble:
        root["text"] = preamble

    # Stack contains the current semantic path.
    #
    # Example:
    #
    # [
    #     {"level": 1, "node": ...},
    #     {"level": 2, "node": ...},
    # ]
    #
    stack = []

    for index, match in enumerate(matches):

        heading_marker = match.group(1)
        heading_text = match.group(2).strip()

        # Actual Markdown heading level.
        heading_level = len(heading_marker)

        # Content belongs to this heading until
        # the next heading.
        content_start = match.end()

        if index + 1 < len(matches):
            content_end = matches[index + 1].start()
        else:
            content_end = len(text)

        content = text[content_start:content_end].strip()

        # Find nearest previous heading with a
        # smaller Markdown heading level.
        while stack and stack[-1]["level"] >= heading_level:
            stack.pop()

        # Parent is either:
        #
        # 1. nearest smaller heading
        # 2. root if no such heading exists
        if stack:
            parent = stack[-1]["node"]
            depth = parent["depth"] + 1
        else:
            parent = root
            depth = 0

        node = {
            "text": heading_text,
            "depth": depth,
            "heading_level": heading_level,
            "children": [],
        }

        if content:
            node["content"] = content

        parent["children"].append(node)

        # Current node becomes the active heading.
        stack.append({
            "level": heading_level,
            "node": node,
        })

    return root


In [ ]:
# build_tree(text=markdown, markers=standard_markers)
build_tree(text=markdown)